<a href="https://colab.research.google.com/github/remyaP12/labcycle_3sem/blob/main/12labcycle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

12.Generate synthetic tabular data using GANs or VAEs, and evaluate its significance using statistical metrics or downstream classifiers.

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [4]:
data = load_breast_cancer()
X = data.data
y = data.target

X = (X - X.mean(axis=0)) / X.std(axis=0)


In [5]:
input_dim = X.shape[1]
latent_dim = 10

encoder_inputs = layers.Input(shape=(input_dim,))
h = layers.Dense(32, activation="relu")(encoder_inputs)
z_mean = layers.Dense(latent_dim)(h)
z_log_var = layers.Dense(latent_dim)(h)

def sampling(args):
    z_mean, z_log_var = args
    eps = tf.random.normal(shape=tf.shape(z_mean))
    return z_mean + tf.exp(0.5 * z_log_var) * eps

z = layers.Lambda(sampling)([z_mean, z_log_var])

encoder = Model(encoder_inputs, [z_mean, z_log_var, z])


In [6]:
latent_inputs = layers.Input(shape=(latent_dim,))
h_dec = layers.Dense(32, activation="relu")(latent_inputs)
decoder_outputs = layers.Dense(input_dim)(h_dec)

decoder = Model(latent_inputs, decoder_outputs)


In [7]:
class VAE(Model):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        if isinstance(data, tuple):
            data = data[0]

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)

            recon_loss = tf.reduce_mean(
                tf.reduce_sum(tf.square(data - reconstruction), axis=1)
            )

            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                    axis=1,
                )
            )

            total_loss = recon_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        return {"loss": total_loss}


In [8]:
vae = VAE(encoder, decoder)
vae.compile(optimizer="adam")
vae.fit(X, epochs=30, batch_size=64, verbose=1)


Epoch 1/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 38.4253
Epoch 2/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 34.1269 
Epoch 3/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 31.9020 
Epoch 4/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 30.1207 
Epoch 5/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 28.6582 
Epoch 6/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 27.6227 
Epoch 7/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 26.1400 
Epoch 8/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 24.5801 
Epoch 9/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 23.3689 
Epoch 10/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 22.0189 
Epoch 11/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 21.1807 
Epoch 12/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 20.5163 
Epoch 13/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 19.6875 
Epoch 14/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 18.7538 
Epoch 15/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 18.3932 
Epoch 16/30
9/9 ━━━━

In [9]:
z_samples = np.random.normal(size=(1000, latent_dim))
X_synthetic = decoder.predict(z_samples)


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [10]:
# Generate synthetic samples equal to real data size
z_samples = np.random.normal(size=(len(y), latent_dim))
X_synthetic = decoder.predict(z_samples)

# Train-test split (lengths now MATCH)
X_train, X_test, y_train, y_test = train_test_split(
    X_synthetic, y, test_size=0.3, random_state=42
)

# Train downstream classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Predict & evaluate
y_pred = clf.predict(X_test)
print("Accuracy on Synthetic Data:", accuracy_score(y_test, y_pred))


18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Accuracy on Synthetic Data: 0.6198830409356725
